# Chunking Experiments

This notebook lets you interactively experiment with different chunking strategies.

**Goal:** Find the optimal chunk size and overlap for your documents.

**Why this matters:**
- Chunks too small → lose context, answers feel incomplete
- Chunks too large → exceed LLM context window, dilute search relevance
- Wrong overlap → lose information at chunk boundaries

## Setup
```bash
pip install matplotlib pandas
```

In [ ]:
# ── IMPORTS ──────────────────────────────────────────────
import sys
sys.path.append('..')   # Add project root to path so we can import our modules

import pandas as pd
import matplotlib.pyplot as plt
from data_preprocessing.chunking.boundary_detector import BoundaryDetector
from data_preprocessing.chunking.table_preserver import TablePreserver
from data_preprocessing.chunking.heading_detector import HeadingDetector
from data_preprocessing.restructuring.structure_analyzer import StructureAnalyzer

print('✓ Imports successful')

## Step 1: Load a Sample Document

In [ ]:
# Load a real document from your corpus
# Replace this with the path to one of YOUR documents

SAMPLE_TEXT = """
# Annual Report 2024

## Executive Summary
Our company achieved record revenue of $12.5 billion in fiscal year 2024,
representing a 23% increase over the previous year. This growth was driven
primarily by strong performance in our cloud services division.

## Financial Highlights

| Metric          | 2024    | 2023    | Change  |
|-----------------|---------|---------|----------|
| Revenue         | $12.5B  | $10.2B  | +23%    |
| Net Income      | $2.1B   | $1.8B   | +17%    |
| EBITDA          | $3.4B   | $2.9B   | +17%    |
| Headcount       | 45,200  | 41,000  | +10%    |

## Product Performance
Our flagship cloud platform grew 45% year over year, adding 12,000 new
enterprise customers. The platform now serves over 80,000 organizations
across 140 countries.

Mobile revenue increased 31% driven by improved user retention rates.
Monthly active users reached 220 million, up from 168 million in 2023.

## Outlook
For fiscal 2025, we project revenue between $14.8B and $15.2B. We plan
to invest $1.2B in R&D and expand our data center infrastructure by 30%.
Hiring will focus on AI engineers and data scientists.
"""

print(f'Document length: {len(SAMPLE_TEXT)} characters')
print(f'Word count: ~{len(SAMPLE_TEXT.split())} words')
print(f'Lines: {len(SAMPLE_TEXT.splitlines())}')

## Step 2: Analyze Document Structure

In [ ]:
# Run the Structure Analyzer
analyzer = StructureAnalyzer()
parsed = {'content': SAMPLE_TEXT, 'source': 'sample.pdf'}
structured = analyzer.analyze(parsed)

print(f'Headings found:  {len(structured["headings"])}')
print(f'Tables found:    {len(structured["tables"])}')
print(f'Sections found:  {len(structured["sections"])}')

print('\nHeadings:')
for h in structured['headings']:
    indent = '  ' * (h['level'] - 1)
    print(f'  {indent}[H{h["level"]}] {h["text"]}')

## Step 3: Compare Chunk Sizes

This is the most important experiment. We test different chunk sizes and measure:
- How many chunks are produced
- Average chunk size
- Minimum and maximum chunk sizes

In [ ]:
def analyze_chunking(text, chunk_size, chunk_overlap):
    """Run chunking and return statistics."""
    chunker = BoundaryDetector(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    chunks = chunker.split(text)
    
    sizes = [len(c) for c in chunks]
    return {
        'chunk_size': chunk_size,
        'overlap': chunk_overlap,
        'num_chunks': len(chunks),
        'avg_size': sum(sizes) / len(sizes) if sizes else 0,
        'min_size': min(sizes) if sizes else 0,
        'max_size': max(sizes) if sizes else 0,
        'chunks': chunks
    }

# Test different configurations
configs = [
    (128,  20),    # Very small chunks
    (256,  40),    # Small chunks
    (512,  50),    # Medium chunks (default)
    (1024, 100),   # Large chunks
    (2048, 200),   # Very large chunks
]

results = [analyze_chunking(SAMPLE_TEXT, size, overlap) 
           for size, overlap in configs]

# Display as a table
df = pd.DataFrame(results).drop('chunks', axis=1)
df.columns = ['Chunk Size (tokens)', 'Overlap', 'Num Chunks', 
              'Avg Chars', 'Min Chars', 'Max Chars']
df = df.round(0).astype(int)
print(df.to_string(index=False))

In [ ]:
# Visualize the chunk size distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Number of chunks per configuration
labels = [f'{r["chunk_size"]} tok' for r in results]
num_chunks = [r['num_chunks'] for r in results]
axes[0].bar(labels, num_chunks, color='steelblue', alpha=0.8)
axes[0].set_title('Number of Chunks by Chunk Size')
axes[0].set_xlabel('Chunk Size (tokens)')
axes[0].set_ylabel('Number of Chunks')
axes[0].grid(axis='y', alpha=0.3)

# Plot 2: Average chunk character length
avg_sizes = [r['avg_size'] for r in results]
axes[1].bar(labels, avg_sizes, color='coral', alpha=0.8)
axes[1].set_title('Average Chunk Length (characters)')
axes[1].set_xlabel('Chunk Size (tokens)')
axes[1].set_ylabel('Average Characters')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('chunking_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved: chunking_comparison.png')

## Step 4: Inspect Individual Chunks

Visually inspect the chunks to make sure they make sense.

In [ ]:
# Inspect chunks from the default configuration (512 tokens)
default_result = next(r for r in results if r['chunk_size'] == 512)

print(f'Total chunks: {default_result["num_chunks"]}')
print('='*60)

for i, chunk in enumerate(default_result['chunks']):
    print(f'\n── Chunk {i+1} ({len(chunk)} chars) ──')
    # Show first 300 chars of each chunk
    preview = chunk[:300]
    if len(chunk) > 300:
        preview += '...'
    print(preview)

## Step 5: Table Preservation Test

Verify that the TablePreserver correctly extracts tables before chunking.

In [ ]:
preserver = TablePreserver()
modified_text, tables = preserver.extract_tables(SAMPLE_TEXT)

print(f'Tables found: {len(tables)}')
for i, table in enumerate(tables):
    print(f'\nTable {i+1} ({table["row_count"]} rows):')
    print(table['text'][:400])
    
print('\n' + '='*60)
print(f'Placeholders in modified text: {modified_text.count("[[TABLE_")}')

# Verify the table content is NOT in the modified text
if tables:
    table_word = tables[0]['text'].split('|')[1].strip()   # First column name
    in_modified = table_word in modified_text
    print(f'Table content still in text: {in_modified} (should be False)')

## Conclusion: Recommended Configuration

Based on the experiments above, fill in your findings:

In [ ]:
# ── YOUR FINDINGS ──────────────────────────────────────────────
# Update these based on your experiments

RECOMMENDED_CONFIG = {
    'chunk_size':    512,   # Update based on your document types
    'chunk_overlap': 50,    # ~10% of chunk_size is a good rule of thumb
    'min_chunk_size': 100,  # Discard chunks smaller than this
}

print('Recommended configuration:')
for key, value in RECOMMENDED_CONFIG.items():
    print(f'  {key}: {value}')

print('\nTo apply these settings, update config/settings.yaml:')
print('preprocessing:')
for key, value in RECOMMENDED_CONFIG.items():
    print(f'  {key}: {value}')